# 천안시 500m 격자 K-Means 클러스터링

이 노트북은 `cheonan_sgis_500m_final_features_with_population.csv`를 이용해 6개 지수를 만들고, K-Means로 4개 클러스터를 생성한 뒤 지도 위에 색상으로 시각화합니다.

입력 파일 위치:

`/content/drive/MyDrive/Cheonan/grid_outputs/cheonan_sgis_500m_final_features_with_population.csv`

결과 저장 위치:

`/content/drive/MyDrive/Cheonan/grid_outputs`

In [6]:
# Colab 기본 준비
!pip -q install scikit-learn folium pyproj

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')

BASE_DIR = Path('/content/drive/MyDrive/Cheonan')
OUTPUT_DIR = BASE_DIR / 'grid_outputs'
INPUT_CSV = OUTPUT_DIR / 'cheonan_sgis_500m_final_features_with_population.csv'

CLUSTERED_CSV = OUTPUT_DIR / 'cheonan_sgis_500m_kmeans_6index_clustered.csv'
SUMMARY_CSV = OUTPUT_DIR / 'cheonan_sgis_500m_kmeans_6index_cluster_summary.csv'
METRICS_CSV = OUTPUT_DIR / 'cheonan_sgis_500m_kmeans_6index_k_metrics.csv'
PCA_CSV = OUTPUT_DIR / 'cheonan_sgis_500m_kmeans_6index_pca.csv'
MAP_HTML = OUTPUT_DIR / 'cheonan_sgis_500m_kmeans_6index_cluster_map.html'

if not INPUT_CSV.exists():
    raise FileNotFoundError(f'입력 파일이 없습니다: {INPUT_CSV}')

df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
print(df.shape)
df.head()

(2552, 207)


,GRID_CD,x_min,y_min,x_max,y_max,center_x,center_y,grid_m,selection,center_lon,...,school_각종학교,school_고등학교,school_유치원,school_중학교,school_초등학교,school_특수학교,theater_영업_정상,theater_폐업,underpass_지하도,underpass_지하차도
0,다바86b64a,986500.0,1864000.0,987000.0,1864500.0,986750.0,1864250.0,500,official_sgis_100m_center_in_cheonan,127.351517,...,0,0,0,0,0,0,0,0,0,0
1,다바65a68b,965000.0,1868500.0,965500.0,1869000.0,965250.0,1868750.0,500,official_sgis_100m_center_in_cheonan,127.110382,...,0,0,0,0,0,0,0,0,0,0
2,다바80a63a,980000.0,1863000.0,980500.0,1863500.0,980250.0,1863250.0,500,official_sgis_100m_center_in_cheonan,127.278703,...,0,0,0,0,0,0,0,0,0,0
3,다바74b70a,974500.0,1870000.0,975000.0,1870500.0,974750.0,1870250.0,500,official_sgis_100m_center_in_cheonan,127.216845,...,0,0,0,0,0,0,0,0,0,0
4,다바77a64a,977000.0,1864000.0,977500.0,1864500.0,977250.0,1864250.0,500,official_sgis_100m_center_in_cheonan,127.245059,...,0,0,0,0,0,0,0,0,0,0


## 1. 6개 지수 정의

각 원자료 count 변수는 다음 순서로 지수화합니다.

`원자료 count → log1p 변환 → 표준화 → 카테고리별 평균`

최종 클러스터링 변수는 다음 6개입니다.

- `population_demand_index`: 인구수요
- `life_welfare_index`: 생활복지
- `commerce_index`: 상업
- `medical_index`: 의료
- `education_index`: 교육
- `leisure_index`: 여가

In [8]:
INDEX_COLUMNS = {
    'population_demand_index': [
        'pop_2026_est_int',
        'households_2026_est_int',
    ],
    'life_welfare_index': [
        'admin_agency_count',
        'elderly_welfare_count',
        'outdoor_shelter_count',
        'underpass_count',
        'shared_medical_equipment_count',
    ],
    'commerce_index': [
        'commerce_total',
        'commerce_major_소매',
        'commerce_major_음식',
        'commerce_major_숙박',
        'good_price_shop_count',
    ],
    'medical_index': [
        'hospital_total',
        'pharmacy_total',
        'hospital_doctor_sum',
        'hospital_type_종합병원',
        'hospital_type_상급종합',
        'hospital_type_병원',
        'hospital_type_보건소',
        'hospital_type_보건지소',
        'hospital_type_보건진료소',
    ],
    'education_index': [
        'school_count',
        'school_유치원',
        'school_초등학교',
        'school_중학교',
        'school_고등학교',
        'school_특수학교',
        'commerce_major_교육',
        'commerce_middle_일반_교육',
        'commerce_middle_기타_교육',
        'commerce_middle_교육_지원',
    ],
    'leisure_index': [
        'theater_count',
        'commerce_major_예술_스포츠',
        'commerce_middle_스포츠_서비스',
        'commerce_middle_유원지_오락',
        'outdoor_shelter_count',
    ],
}

RAW_SUM_COLUMNS = {
    'population_demand_raw': ['pop_2026_est_int', 'households_2026_est_int'],
    'life_welfare_raw': [
        'admin_agency_count',
        'elderly_welfare_count',
        'outdoor_shelter_count',
        'underpass_count',
        'shared_medical_equipment_count',
    ],
    'commerce_raw': [
        'commerce_total',
        'commerce_major_소매',
        'commerce_major_음식',
        'commerce_major_숙박',
        'good_price_shop_count',
    ],
    'medical_raw': ['hospital_total', 'pharmacy_total', 'hospital_doctor_sum'],
    'education_raw': [
        'school_count',
        'commerce_major_교육',
        'commerce_middle_일반_교육',
        'commerce_middle_기타_교육',
        'commerce_middle_교육_지원',
    ],
    'leisure_raw': [
        'theater_count',
        'commerce_major_예술_스포츠',
        'commerce_middle_스포츠_서비스',
        'commerce_middle_유원지_오락',
        'outdoor_shelter_count',
    ],
}

PALETTE = {
    1: '#d73027',
    2: '#1a9850',
    3: '#4575b4',
    4: '#984ea3',
}

CLUSTER_NAMES = {
    1: '중심상업·서비스형',
    2: '고밀주거·상업교육형',
    3: '일반주거·시설부족형',
    4: '외곽저밀도형',
}

In [9]:
def require_columns(data: pd.DataFrame, columns: list[str]) -> None:
    missing = [col for col in columns if col not in data.columns]
    if missing:
        raise ValueError(f'누락된 컬럼: {missing}')


def zscore_log_columns(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    require_columns(data, columns)
    values = data[columns].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    values = values.clip(lower=0.0)
    logged = np.log1p(values)
    scaled = StandardScaler().fit_transform(logged)
    return pd.DataFrame(scaled, columns=columns, index=data.index)


def add_indices(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    data = data.copy()
    index_meta_rows = []

    for index_name, columns in INDEX_COLUMNS.items():
        scaled_components = zscore_log_columns(data, columns)
        component_cols = []

        for col in columns:
            component_col = f'{index_name}__{col}'
            data[component_col] = scaled_components[col]
            component_cols.append(component_col)

        data[index_name] = data[component_cols].mean(axis=1)
        index_meta_rows.append({
            'index_name': index_name,
            'source_columns': ', '.join(columns),
            'component_columns': ', '.join(component_cols),
        })

    for raw_name, columns in RAW_SUM_COLUMNS.items():
        require_columns(data, columns)
        data[raw_name] = data[columns].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1)

    supply_raw_cols = [
        'life_welfare_raw',
        'commerce_raw',
        'medical_raw',
        'education_raw',
        'leisure_raw',
    ]
    data['facility_diversity'] = (data[supply_raw_cols] > 0).sum(axis=1)
    return data, pd.DataFrame(index_meta_rows)


def order_clusters(data: pd.DataFrame, raw_label_col: str = 'cluster_raw') -> pd.Series:
    index_cols = list(INDEX_COLUMNS)
    profile = data.groupby(raw_label_col)[index_cols].mean()
    service_cols = [
        'life_welfare_index',
        'commerce_index',
        'medical_index',
        'education_index',
        'leisure_index',
    ]
    profile['service_mean'] = profile[service_cols].mean(axis=1)
    profile['center_service_score'] = (
        profile['commerce_index'] + profile['medical_index'] + profile['life_welfare_index']
    )
    profile['low_density_score'] = profile['population_demand_index'] + profile['service_mean']
    profile['education_residential_score'] = (
        profile['population_demand_index'] + profile['education_index']
    )

    remaining = set(profile.index)
    ordered = {}

    center_raw = profile.loc[list(remaining), 'center_service_score'].idxmax()
    ordered[center_raw] = 1
    remaining.remove(center_raw)

    low_raw = profile.loc[list(remaining), 'low_density_score'].idxmin()
    ordered[low_raw] = 4
    remaining.remove(low_raw)

    if remaining:
        edu_raw = profile.loc[list(remaining), 'education_residential_score'].idxmax()
        ordered[edu_raw] = 2
        remaining.remove(edu_raw)

    for raw in remaining:
        ordered[raw] = 3

    return data[raw_label_col].map(ordered).astype(int)


def profile_text(row: pd.Series) -> str:
    values = {
        '인구': row['population_demand_index_mean'],
        '생활': row['life_welfare_index_mean'],
        '상업': row['commerce_index_mean'],
        '의료': row['medical_index_mean'],
        '교육': row['education_index_mean'],
        '여가': row['leisure_index_mean'],
    }
    if max(values.values()) < 0:
        least_low = sorted(values, key=values.get, reverse=True)[:2]
        return f'전반적 저밀도 / 상대적으로 덜 낮음: {", ".join(least_low)}'

    top = sorted(values, key=values.get, reverse=True)[:2]
    low = sorted(values, key=values.get)[:2]
    return f'강점: {", ".join(top)} / 약점: {", ".join(low)}'

## 2. K-Means 실행

정책적으로 4개 유형을 목표로 하되, `k=3~6`의 군집 품질 지표도 함께 계산합니다.

In [10]:
df_indexed, index_meta = add_indices(df)
index_cols = list(INDEX_COLUMNS)

x = StandardScaler().fit_transform(df_indexed[index_cols])

metrics = []
for k in range(3, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=30)
    labels = model.fit_predict(x)
    metrics.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette': silhouette_score(x, labels),
        'calinski_harabasz': calinski_harabasz_score(x, labels),
        'davies_bouldin': davies_bouldin_score(x, labels),
    })

metrics_df = pd.DataFrame(metrics)
display(metrics_df)

final_model = KMeans(n_clusters=4, random_state=42, n_init=50)
df_indexed['cluster_raw'] = final_model.fit_predict(x)
df_indexed['cluster'] = order_clusters(df_indexed)
df_indexed['cluster_name'] = df_indexed['cluster'].map(CLUSTER_NAMES)
df_indexed['cluster_color'] = df_indexed['cluster'].map(PALETTE)

pca = PCA(n_components=2, random_state=42)
pca_values = pca.fit_transform(x)
df_indexed['pca1'] = pca_values[:, 0]
df_indexed['pca2'] = pca_values[:, 1]

print(df_indexed['cluster_name'].value_counts())
df_indexed[['GRID_CD', 'center_adm_nm', *index_cols, 'cluster', 'cluster_name']].head()

,k,inertia,silhouette,calinski_harabasz,davies_bouldin
0,3,6066.495414,0.680087,1942.372786,1.066029
1,4,5025.537620,0.440408,1738.447912,1.133685
2,5,4459.211306,0.438639,1549.716862,1.242382
3,6,4054.571132,0.439648,1413.782763,1.263044


cluster_name
외곽저밀도형        1390
일반주거·시설부족형     958
고밀주거·상업교육형     142
중심상업·서비스형       62
Name: count, dtype: int64


,GRID_CD,center_adm_nm,population_demand_index,life_welfare_index,commerce_index,medical_index,education_index,leisure_index,cluster,cluster_name
0,다바86b64a,동면,0.566387,0.347901,-0.156563,-0.102061,-0.180378,0.276310,3,일반주거·시설부족형
1,다바65a68b,불당1동,1.820968,5.020365,4.065964,3.934787,3.363874,3.799874,1,중심상업·서비스형
2,다바80a63a,병천면,0.589609,-0.159890,0.882875,-0.102061,0.532834,0.877284,3,일반주거·시설부족형
3,다바74b70a,목천읍,-1.059785,-0.159890,-0.400309,-0.102061,-0.180378,-0.231481,4,외곽저밀도형
4,다바77a64a,목천읍,1.091987,-0.159890,0.712488,-0.102061,0.062446,-0.231481,3,일반주거·시설부족형


In [11]:
summary = (
    df_indexed.groupby(['cluster', 'cluster_name'], as_index=False)
    .agg(
        grid_count=('GRID_CD', 'count'),
        population_sum=('pop_2026_est_int', 'sum'),
        households_sum=('households_2026_est_int', 'sum'),
        life_welfare_raw_mean=('life_welfare_raw', 'mean'),
        commerce_raw_mean=('commerce_raw', 'mean'),
        medical_raw_mean=('medical_raw', 'mean'),
        education_raw_mean=('education_raw', 'mean'),
        leisure_raw_mean=('leisure_raw', 'mean'),
        facility_diversity_mean=('facility_diversity', 'mean'),
        population_demand_index_mean=('population_demand_index', 'mean'),
        life_welfare_index_mean=('life_welfare_index', 'mean'),
        commerce_index_mean=('commerce_index', 'mean'),
        medical_index_mean=('medical_index', 'mean'),
        education_index_mean=('education_index', 'mean'),
        leisure_index_mean=('leisure_index', 'mean'),
    )
    .sort_values('cluster')
)

summary['grid_share'] = summary['grid_count'] / len(df_indexed)
summary['population_share'] = summary['population_sum'] / df_indexed['pop_2026_est_int'].sum()
summary['profile_note'] = summary.apply(profile_text, axis=1)

display(summary)

,cluster,cluster_name,grid_count,population_sum,households_sum,life_welfare_raw_mean,commerce_raw_mean,medical_raw_mean,education_raw_mean,leisure_raw_mean,facility_diversity_mean,population_demand_index_mean,life_welfare_index_mean,commerce_index_mean,medical_index_mean,education_index_mean,leisure_index_mean,grid_share,population_share,profile_note
0,1,중심상업·서비스형,62,226076,110457,4.548387,496.967742,45.338710,54.080645,33.161290,4.870968,2.437828,2.538209,3.415250,2.378025,2.437256,3.512366,0.024295,0.339526,"강점: 여가, 상업 / 약점: 의료, 교육"
1,2,고밀주거·상업교육형,142,314223,147069,1.809859,120.084507,2.873239,14.169014,7.732394,3.922535,1.992297,0.771656,1.594475,0.390850,1.374940,1.250829,0.055643,0.471907,"강점: 인구, 상업 / 약점: 의료, 생활"
2,3,일반주거·시설부족형,958,121656,61675,0.200418,8.235908,0.104384,0.308977,0.402923,1.293319,0.584891,-0.053845,0.065765,-0.065259,-0.105576,-0.089225,0.375392,0.182706,"강점: 인구, 상업 / 약점: 교육, 여가"
3,4,외곽저밀도형,1390,3903,2170,0.012950,0.363309,0.089928,0.023022,0.025899,0.166187,-0.715379,-0.154936,-0.360549,-0.101021,-0.176410,-0.222955,0.544671,0.005862,"전반적 저밀도 / 상대적으로 덜 낮음: 의료, 생활"


## 3. 결과 저장

저장되는 주요 파일:

- `cheonan_sgis_500m_kmeans_6index_clustered.csv`
- `cheonan_sgis_500m_kmeans_6index_cluster_summary.csv`
- `cheonan_sgis_500m_kmeans_6index_k_metrics.csv`
- `cheonan_sgis_500m_kmeans_6index_cluster_map.html`

In [12]:
cluster_cols = [
    'GRID_CD',
    'center_adm_cd',
    'center_adm_nm',
    'center_lon',
    'center_lat',
    'x_min',
    'y_min',
    'x_max',
    'y_max',
    'pop_2026_est_int',
    'households_2026_est_int',
    'population_demand_index',
    'life_welfare_index',
    'commerce_index',
    'medical_index',
    'education_index',
    'leisure_index',
    'life_welfare_raw',
    'commerce_raw',
    'medical_raw',
    'education_raw',
    'leisure_raw',
    'facility_diversity',
    'cluster',
    'cluster_name',
    'cluster_color',
    'pca1',
    'pca2',
]

df_indexed[cluster_cols].to_csv(CLUSTERED_CSV, index=False, encoding='utf-8-sig')
summary.to_csv(SUMMARY_CSV, index=False, encoding='utf-8-sig')
metrics_df.to_csv(METRICS_CSV, index=False, encoding='utf-8-sig')
index_meta.to_csv(PCA_CSV, index=False, encoding='utf-8-sig')

print('저장 완료')
print(CLUSTERED_CSV)
print(SUMMARY_CSV)
print(METRICS_CSV)

저장 완료
/content/drive/MyDrive/Cheonan/grid_outputs/cheonan_sgis_500m_kmeans_6index_clustered.csv
/content/drive/MyDrive/Cheonan/grid_outputs/cheonan_sgis_500m_kmeans_6index_cluster_summary.csv
/content/drive/MyDrive/Cheonan/grid_outputs/cheonan_sgis_500m_kmeans_6index_k_metrics.csv


## 4. 지도 시각화

`pyproj`로 EPSG:5179 좌표를 위경도 좌표로 변환한 뒤, `folium`으로 각 격자를 클러스터 색상으로 칠합니다.

In [13]:
import folium
from pyproj import Transformer

transformer = Transformer.from_crs('EPSG:5179', 'EPSG:4326', always_xy=True)


def grid_polygon_latlon(row: pd.Series) -> list[tuple[float, float]]:
    xs = [row['x_min'], row['x_max'], row['x_max'], row['x_min'], row['x_min']]
    ys = [row['y_min'], row['y_min'], row['y_max'], row['y_max'], row['y_min']]
    lons, lats = transformer.transform(xs, ys)
    return [(float(lat), float(lon)) for lat, lon in zip(lats, lons)]


center = [float(df_indexed['center_lat'].mean()), float(df_indexed['center_lon'].mean())]
m = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')

for _, row in df_indexed.iterrows():
    popup_html = f'''
    <b>{row["GRID_CD"]}</b><br>
    클러스터: {int(row["cluster"])}. {row["cluster_name"]}<br>
    행정동: {row["center_adm_nm"]}<br>
    추정 인구: {int(row["pop_2026_est_int"]):,}명<br>
    인구수요: {row["population_demand_index"]:.3f}<br>
    생활복지: {row["life_welfare_index"]:.3f}<br>
    상업: {row["commerce_index"]:.3f}<br>
    의료: {row["medical_index"]:.3f}<br>
    교육: {row["education_index"]:.3f}<br>
    여가: {row["leisure_index"]:.3f}
    '''
    folium.Polygon(
        locations=grid_polygon_latlon(row),
        color=row['cluster_color'],
        weight=0.4,
        fill=True,
        fill_color=row['cluster_color'],
        fill_opacity=0.58,
        popup=folium.Popup(popup_html, max_width=320),
    ).add_to(m)

legend_items = ''.join(
    f'''
    <div style="display:flex; align-items:center; gap:6px; margin:4px 0;">
      <span style="display:inline-block; width:13px; height:13px; background:{PALETTE[int(r.cluster)]}; border:1px solid #555;"></span>
      <span>{int(r.cluster)}. {r.cluster_name} ({int(r.grid_count):,}개)</span>
    </div>
    '''
    for _, r in summary.iterrows()
)

legend_html = f'''
<div style="
    position: fixed;
    bottom: 24px;
    right: 24px;
    z-index: 9999;
    background: rgba(255,255,255,0.94);
    border: 1px solid #bbb;
    padding: 12px 14px;
    font-size: 13px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.18);
">
  <div style="font-weight:700; margin-bottom:8px;">K-Means 4개 클러스터</div>
  {legend_items}
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))
m.save(MAP_HTML)
print(MAP_HTML)
m

Output hidden; open in https://colab.research.google.com to view.